**Amaliyot: ML - Toshkent uylarining narxini bashorat qilish.**

# Ustunlar ta'rifi
- `location` - sotilayotgan uy manzili
- `district` - uy joylashgan tuman
- `rooms` - xonalar soni
- `size` - uy maydoni (kv.m)
- `level` - uy joylashgan qavat
- `max_levels` - ja'mi qavatlar soni
- `price` - uy narxi


In [67]:
# Kutubxonalar

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [55]:
# Data

df = pd.read_csv(
    'https://raw.githubusercontent.com/anvarnarz/praktikum_datasets/main/housing_data_08-02-2021.csv'
)


In [56]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7565 entries, 0 to 7564
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   location    7565 non-null   object
 1   district    7565 non-null   object
 2   rooms       7565 non-null   int64 
 3   size        7565 non-null   object
 4   level       7565 non-null   int64 
 5   max_levels  7565 non-null   int64 
 6   price       7565 non-null   object
dtypes: int64(3), object(4)
memory usage: 413.8+ KB


In [73]:
# Noto‘g‘ri qiymatlarni olib tashlaymiz
df.drop(df[df['price'] == 'Договорная'].index, inplace=True)
df.drop(df[df['size'] == 'Площадьземли:1сот'].index, inplace=True)

# Type o‘zgartirish
df['price'] = df['price'].astype(int)
df['size'] = df['size'].astype(float)


In [77]:
# Train , Test set

train_set, test_set = train_test_split(df, test_size=0.2, random_state=42)

X_train = train_set.drop('price', axis=1)
y_train = train_set['price']

X_test = test_set.drop('price', axis=1)
y_test = test_set['price']


In [91]:
num_attribs = ['rooms','size','level','max_levels']
cat_attribs = ['district']

full_pipeline = ColumnTransformer([
    ('num', StandardScaler(), num_attribs),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_attribs)
])

# X_train ustunlari bilan fit_transform qilamiz
X_train_prepared = full_pipeline.fit_transform(X_train)
X_test_prepared = full_pipeline.transform(X_test)


In [92]:
# Model

LR_model = LinearRegression()
LR_model.fit(X_train_prepared, y_train)


LinearRegression()

In [93]:
# MAE, RMSE

y_pred = LR_model.predict(X_test_prepared)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("MAE:", mae)
print("RMSE:", rmse)


MAE: 67771.32084531563
RMSE: 1366742.755611274


In [94]:
# Test

sample_data = X_test.sample(10, random_state=42)
sample_labels = y_test.loc[sample_data.index]

sample_prepared = full_pipeline.transform(sample_data)
sample_pred = LR_model.predict(sample_prepared)

pd.DataFrame({
    'Real': sample_labels.values,
    'Bashorat': sample_pred.astype(int)
})


,Real,Bashorat
0,51000,57791
1,63000,74656
2,40000,40789
3,40000,29045
4,66000,94777
5,26200,14432
6,38000,57151
7,110000,92259
8,46835,50647
9,35000,35183


In [95]:
# Model

RF_model = RandomForestRegressor()
RF_model.fit(X_train_prepared, y_train)


RandomForestRegressor()

In [96]:
# Test

sample_data = X_test.sample(10, random_state=42)
sample_labels = y_test.loc[sample_data.index]

sample_prepared = full_pipeline.transform(sample_data)
sample_pred = RF_model.predict(sample_prepared)

pd.DataFrame({
    'Real': sample_labels.values,
    'Bashorat': sample_pred.astype(int)
})


,Real,Bashorat
0,51000,60915
1,63000,60572
2,40000,34461
3,40000,36624
4,66000,62396
5,26200,24165
6,38000,38676
7,110000,86956
8,46835,50280
9,35000,42679


In [97]:
# MAE , RMSE

y_pred = RF_model.predict(X_test_prepared)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("MAE:", mae)
print("RMSE:", rmse)


MAE: 60307.09821722107
RMSE: 1370379.9576057598


In [100]:
# Modelni baholash

from sklearn.model_selection import cross_val_score

X = df.drop('price',axis=1)
y = df['price'].copy()

X_prepared = full_pipeline.transform(X)

mse_scores = cross_val_score(LR_model, X_prepared, y, scoring='neg_mean_squared_error', cv=5)

rmse_scores = np.sqrt(-mse_scores)

print("RMSE scores:", rmse_scores)

scores_rf = cross_val_score(RF_model, X_prepared, y, scoring='neg_mean_squared_error', cv=5)

rmse_scores_rf = np.sqrt(-mse_scores)

print("RMSE scores:", rmse_scores_rf)


RMSE scores: [  64179.93212171  106766.74216729   47264.63386425 1344488.91872085
  474807.83580356]
RMSE scores: [  64179.93212171  106766.74216729   47264.63386425 1344488.91872085
  474807.83580356]


In [101]:
# Modelni saqlash

import pickle

filename = 'LR_model.pkl'
pickle.dump(LR_model, open(filename, 'wb'))